In [ ]:
import os, sys, torch, random, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, r"c:\repos\DroneDetectionRF")
from NoisyUAV.funciones.dataset.cargador import cargar_muestra
from NoisyUAV.funciones.dsp_rf.detector_entropia import detectar_bursts, plot_muestra, print_diagnostico
from NoisyUAV.modelo_alumn_v2_dual.model import DualStreamCVCNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
ruta_pesos = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_v2_dual\checkpoints\best_model.pth"
model = DualStreamCVCNN().to(device)
ckpt = torch.load(ruta_pesos, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"✅ Modelo Dual-Stream V2 cargado en {device.type.upper()}")
print(f"   Época: {ckpt.get('epoch','N/A')} | Val F1: {ckpt.get('val_f1',0.0):.4f} | Val Acc: {ckpt.get('val_acc',0.0):.4f}")

In [ ]:
# ===== PARÁMETROS =====
TARGET_DESEADO = 0      # 0=DJI, 1=FutabaT14, 2=FutabaT7, 3=Graupner, 4=Ruido, 5=Taranis, 6=Turnigy
SNR_DESEADA    = -14
SOLO_TEST      = True
RUTA_RAW_DIR   = r"C:\TFM_data\NoisyUAV\drone_RF_data"
TARGET_NOISE   = 4      # Target 4 = ficheros de ruido puro
# ======================

csv_path  = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_v2_dual\dataset_v5_pointers.csv"
df_pseudo = pd.read_csv(csv_path)

filtro = (df_pseudo['target_multiclass'] == TARGET_DESEADO) & (df_pseudo['snr'] == SNR_DESEADA)
if SOLO_TEST:
    filtro &= (df_pseudo['split'] == 'test')
df_candidatos = df_pseudo[filtro]

if df_candidatos.empty:
    pattern = re.compile(rf"IQdata_sample\d+_target{TARGET_DESEADO}_snr{SNR_DESEADA}\.pt")
    todos = [f for f in os.listdir(RUTA_RAW_DIR) if pattern.match(f)]
    if not todos:
        raise ValueError(f"❌ No existen ficheros para Target={TARGET_DESEADO} y SNR={SNR_DESEADA} dB.")
    filename_random = random.choice(todos)
    print("⚠️ Fichero no está en el CSV (CFAR ciego a esta SNR). Selección directa del disco.")
else:
    filename_random = random.choice(df_candidatos['filename'].unique())

ruta_completa = os.path.join(RUTA_RAW_DIR, filename_random)
print("=" * 60)
print(f"🎬 Muestra: {filename_random}")
print(f"   Target: {TARGET_DESEADO} | SNR: {SNR_DESEADA} dB | Solo Test: {SOLO_TEST}")
print("=" * 60)

In [ ]:
iq_tensor, _, original_target, original_snr = cargar_muestra(ruta_completa)

# Parámetros IDÉNTICOS a build_dataset_v5_pointers.py
FS      = 14e6
NPERSEG = 2048
Z_THRESH = 3.0

t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=1.3, merge_gap_ms=0.75, min_z_abs=3.5,
    bg_mult=4, max_bins_frac=1.0, smooth_ms=0.3, adaptive_window_ms=15,
)
print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS, z_thresh=Z_THRESH,
    bg_mult=4, max_bins_frac=1.0, min_burst_ms=0.5, merge_gap_ms=0.75,
    target=f"Target {original_target}", snr=f"{original_snr}", index=0,
)
fig_2d = plot_muestra(
    iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    bg_mult=4, max_bins_frac=1.0, adaptive_window_ms=10,
    titulo=f"CFAR: {filename_random}"
)
fig_2d.show()


Nueva forma de veredicto

In [ ]:
# Ground truth del fichero (determinista desde el nombre)
es_dron_real = (original_target != TARGET_NOISE)
gt_str = "DRON" if es_dron_real else "RUIDO"

# Parámetros
PASO_MS = 2.0
window_len = 131072
half_win   = window_len // 2
max_idx    = iq_tensor.shape[1]

global_nf     = float(np.median(nf_v))
global_H_mean = float(np.mean(H_smooth))

# --- FASE 1: Veredicto CFAR ---
print("=" * 60)
print(f"  VEREDICTO DUAL-STREAM | GT: {gt_str} (Target {original_target}, {original_snr} dB)")
print("=" * 60)

centers, z_peaks = [], []
is_blind = False

if len(bursts) > 0:
    for b in bursts:
        centers.append((b['t0'] + b['t1']) / 2.0)
        z_peaks.append(abs(b['z_peak']))
else:
    print("  CFAR ciego. Activando escaner ciego (cada 10ms)...")
    is_blind = True
    L_ms = (iq_tensor.shape[1] / FS) * 1000
    t_s = 5.0
    while t_s < L_ms - 5.0:
        centers.append(t_s); z_peaks.append(0.0); t_s += 10.0

drones_cfar, ruidos_cfar = 0, 0

with torch.no_grad():
    for i, (t_c, z) in enumerate(zip(centers, z_peaks)):
        c_idx = int((t_c / 1000.0) * FS)
        s = c_idx - half_win; e = c_idx + half_win
        if s < 0:         win = iq_tensor[:, 0:window_len]
        elif e > max_idx: win = iq_tensor[:, max_idx-window_len:max_idx]
        else:             win = iq_tensor[:, s:e]
        pwr = win.pow(2).mean().clamp(min=1e-12).sqrt()
        win = (win / pwr).unsqueeze(0).to(device)
        feat = torch.tensor([[global_nf, global_H_mean, float(z)]], dtype=torch.float32).to(device)
        with torch.amp.autocast('cuda'):
            logit, attn = model(win, feat)
            prob = torch.sigmoid(logit).item() * 100
            aw   = attn[0].cpu().numpy() * 100
        etiq = "B" if not is_blind else "W"
        if prob > 50.0:
            drones_cfar += 1
            print(f"  [{etiq}{i+1:02d}] t={t_c:6.2f}ms | DRON   [{prob:5.1f}%] OK | IQ={aw[0]:.0f}% PSD={aw[1]:.0f}%")
        else:
            ruidos_cfar += 1
            print(f"  [{etiq}{i+1:02d}] t={t_c:6.2f}ms | RUIDO  [{prob:5.1f}%]    | IQ={aw[0]:.0f}% PSD={aw[1]:.0f}%")

veredicto_cfar = drones_cfar > 0

# --- FASE 2: Sliding Window ---
paso_idx = int((PASO_MS / 1000.0) * FS)
L_total  = iq_tensor.shape[1]
z_global = max(z_peaks) if z_peaks else 0.0

t_sw, prob_sw, attn_iq_sw, attn_psd_sw = [], [], [], []

with torch.no_grad():
    for v in range((L_total - window_len) // paso_idx + 1):
        s_v = v * paso_idx; e_v = s_v + window_len
        if e_v > L_total: break
        win_v = iq_tensor[:, s_v:e_v].clone()
        pwr   = win_v.pow(2).mean().clamp(min=1e-12).sqrt()
        win_v = (win_v / pwr).unsqueeze(0).to(device)
        feat_v = torch.tensor([[global_nf, global_H_mean, z_global]], dtype=torch.float32).to(device)
        with torch.amp.autocast('cuda'):
            logit_v, attn_v = model(win_v, feat_v)
            p_v  = torch.sigmoid(logit_v).item() * 100
            aw_v = attn_v[0].cpu().numpy() * 100
        t_sw.append(((s_v + window_len / 2) / FS) * 1000)
        prob_sw.append(p_v); attn_iq_sw.append(aw_v[0]); attn_psd_sw.append(aw_v[1])

prob_arr = np.array(prob_sw)
t_arr    = np.array(t_sw)

# --- Métricas de consenso ---
# Umbral duro (>50%): para DRON confirmado
THRESH_HARD_AREA = 0.05   # >5% de ventanas
THRESH_HARD_RUN  = 2      # >=2 ventanas contiguas (>=4ms)

area_score = np.mean(prob_arr > 50)
run, max_run = 0, 0
for p in prob_arr:
    run = run + 1 if p > 50 else 0
    max_run = max(max_run, run)

# Umbral suave (>38%): para SOSPECHOSO — requiere también coherencia temporal
THRESH_SOFT_MAX = 38.0
THRESH_SOFT_RUN = 3       # >=3 ventanas contiguas (>=6ms) por encima del umbral suave

run_soft, max_run_soft = 0, 0
for p in prob_arr:
    run_soft = run_soft + 1 if p > THRESH_SOFT_MAX else 0
    max_run_soft = max(max_run_soft, run_soft)

p_max_sw  = float(np.max(prob_arr))
p_mean_sw = float(np.mean(prob_arr))

# --- FASE 3: Consenso con tres niveles ---
# DRON: CFAR + SW ambos confirman activación fuerte sostenida
if veredicto_cfar and (area_score >= THRESH_HARD_AREA) and (max_run >= THRESH_HARD_RUN):
    nivel    = "DRON"
    correcto = es_dron_real

# SOSPECHOSO: SW ve activación SOSTENIDA por encima del umbral suave
# (un único pico puntual no es suficiente: max_run_soft debe ser >=3)
elif max_run_soft >= THRESH_SOFT_RUN:
    nivel    = "SOSPECHOSO"
    correcto = es_dron_real   # correcto si hay dron real detrás

# RUIDO: sin activación sostenida en ningún umbral
else:
    nivel    = "RUIDO"
    correcto = not es_dron_real

icono = "CORRECTO" if correcto else "INCORRECTO"

print("-" * 60)
print(f"  CFAR solo:    {'DRON' if veredicto_cfar else 'RUIDO'} ({drones_cfar}/{drones_cfar+ruidos_cfar} positivos)")
print(f"  SW duro:      area={area_score:.2f} | racha>{50:.0f}%={max_run}v ({max_run*PASO_MS:.0f}ms)")
print(f"  SW suave:     racha>{THRESH_SOFT_MAX:.0f}%={max_run_soft}v ({max_run_soft*PASO_MS:.0f}ms) | P_max={p_max_sw:.1f}% | P_mean={p_mean_sw:.1f}%")
print(f"  CONSENSO:     [{nivel}]")
print(f"  Ground Truth: {gt_str} (Target {original_target})")
print(f"  RESULTADO:    {icono}")
print("=" * 60)

# --- Gráfica ---
nivel_to_color = {"DRON": "red", "SOSPECHOSO": "orange", "RUIDO": "green"}
title_color = nivel_to_color[nivel]
gt_label = "DRON" if es_dron_real else "RUIDO"

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.fill_between(t_arr, prob_arr, alpha=0.3, color=title_color)
ax1.plot(t_arr, prob_arr, color=title_color, linewidth=1.5, label='P(Dron)')
ax1.axhline(50,             color='gray',   linestyle='--', alpha=0.7, linewidth=1.5,
            label=f'Umbral DRON (50%) — racha>={THRESH_HARD_RUN}v')
ax1.axhline(THRESH_SOFT_MAX, color='orange', linestyle=':', alpha=0.9, linewidth=1.5,
            label=f'Umbral SOSPECHOSO ({THRESH_SOFT_MAX:.0f}%) — racha>={THRESH_SOFT_RUN}v')
ax1.axhline(p_mean_sw,      color='purple', linestyle='-.', alpha=0.5, linewidth=1.0,
            label=f'P_media={p_mean_sw:.1f}%')
ax1.set_ylabel('Probabilidad Dron (%)')
ax1.set_ylim(0, 100)
ax1.set_title(
    f'Sliding Window — {filename_random} | GT: {gt_label} | [{nivel}] [{icono}]',
    color=title_color, fontweight='bold'
)
ax1.legend(fontsize=8, loc='upper right'); ax1.grid(True, alpha=0.3)

# Zonas rojas: activacion dura (>50%)
above_hard = prob_arr > 50
if np.any(above_hard):
    ups   = np.where(np.diff(above_hard.astype(int)) == 1)[0]
    downs = np.where(np.diff(above_hard.astype(int)) == -1)[0]
    for su in ups:
        cands = downs[downs > su]
        if len(cands): ax1.axvspan(t_arr[su], t_arr[cands[0]], alpha=0.25, color='red')

# Zonas naranjas: activacion suave (>THRESH_SOFT_MAX pero <50%)
above_soft = (prob_arr > THRESH_SOFT_MAX) & ~above_hard
if np.any(above_soft):
    ups   = np.where(np.diff(above_soft.astype(int)) == 1)[0]
    downs = np.where(np.diff(above_soft.astype(int)) == -1)[0]
    for su in ups:
        cands = downs[downs > su]
        if len(cands): ax1.axvspan(t_arr[su], t_arr[cands[0]], alpha=0.12, color='orange')

ax2.plot(t_arr, attn_iq_sw,  color='blue',   linewidth=1.2, label='Rama IQ')
ax2.plot(t_arr, attn_psd_sw, color='orange', linewidth=1.2, label='Rama PSD')
ax2.set_ylabel('Peso Atencion (%)'); ax2.set_xlabel('Tiempo (ms)')
ax2.set_ylim(0, 100); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\nSW Stats: {len(prob_arr)} ventanas | P_media={p_mean_sw:.1f}% | P_max={p_max_sw:.1f}% (t={t_arr[np.argmax(prob_arr)]:.1f}ms)")
print(f"   Racha >{50:.0f}%:          {max_run}v  = {max_run*PASO_MS:.0f}ms  [umbral DRON: >={THRESH_HARD_RUN}v]")
print(f"   Racha >{THRESH_SOFT_MAX:.0f}%: {max_run_soft}v  = {max_run_soft*PASO_MS:.0f}ms  [umbral SOSP: >={THRESH_SOFT_RUN}v]")


In [ ]:
print("==================================================")
print("     VEREDICTO DUAL-STREAM (Inferencia en vivo)   ")
print("==================================================")

drones_encontrados = 0
ruidos_encontrados = 0

# Variables físicas globales para este fichero
global_nf = float(np.median(nf_v))
global_H_mean = float(np.mean(H_smooth))

centers = []
z_peaks = []
is_blind = False

# Decidir si usamos el CFAR como puntero o pasamos al Escáner Ciego
if len(bursts) > 0:
    for b in bursts:
        centers.append((b['t0'] + b['t1']) / 2.0)
        z_peaks.append(abs(b['z_peak']))
else:
    print("  ⚠️ El CFAR no ve nada. Activando Escáner Ciego MIL (cada 10ms)...")
    is_blind = True
    L_ms = (iq_tensor.shape[1] / FS) * 1000
    t_scan = 5.0
    while t_scan < L_ms - 5.0:
        centers.append(t_scan)
        z_peaks.append(0.0)
        t_scan += 10.0

# Preparación de variables para el recorte de V2
window_len = 131072
half_win = window_len // 2
max_idx = iq_tensor.shape[1]

with torch.no_grad():
    for i, (t_ms_center, z) in enumerate(zip(centers, z_peaks)):
        # 1. RECORTAR LA ONDA ALREDEDOR DEL CENTRO (Fijo de 9.4ms)
        c_idx = int((t_ms_center / 1000.0) * FS)
        start = c_idx - half_win
        end = c_idx + half_win
        
        if start < 0:
            win = iq_tensor[:, 0:window_len]
        elif end > max_idx:
            win = iq_tensor[:, max_idx-window_len:max_idx]
        else:
            win = iq_tensor[:, start:end]
            
        # [CRÍTICO] NORMALIZACIÓN RMS
        power = win.pow(2).mean().clamp(min=1e-12).sqrt()
        win = win / power
            
        input_ia = win.unsqueeze(0).to(device) # [1, 2, 131072]
        
        # 2. CONSTRUIR EL PERFIL FÍSICO (3 Dimensiones)
        feat_t = torch.tensor([[global_nf, global_H_mean, float(z)]], dtype=torch.float32).to(device)
        
        # 3. CLASIFICACIÓN CON AMP (Auto Mixed Precision)
        with torch.amp.autocast('cuda'):
            logit, attn = model(input_ia, feat_t)
            prob_dron = torch.sigmoid(logit).item() * 100 
            attn_peso = attn[0].cpu().numpy() * 100 # [IQ, PSD]
            
        etiq = "B" if not is_blind else "W" # B=Burst, W=Window
        
        if prob_dron > 50.0:
            drones_encontrados += 1
            print(f"  [{etiq}{i+1:02d}] t={t_ms_center:6.2f} ms | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅ | Foco: IQ={attn_peso[0]:.0f}% PSD={attn_peso[1]:.0f}%")
        else:
            ruidos_encontrados += 1
            print(f"  [{etiq}{i+1:02d}] t={t_ms_center:6.2f} ms | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌ | Foco: IQ={attn_peso[0]:.0f}% PSD={attn_peso[1]:.0f}%")

print("-" * 60)
print(f"RESUMEN FINAL: {drones_encontrados} Drones | {ruidos_encontrados} Ruidos")
if drones_encontrados > 0:
    print(f"Veredicto del Fichero: 🔴 ES UN DRON ({drones_encontrados}/{drones_encontrados+ruidos_encontrados} ventanas positivas)")
else:
    print("Veredicto del Fichero: 🟢 ES RUIDO / SALA VACÍA")


In [ ]:
# CELDA: ESCANEO COMPLETO (Sliding Window) para SNRs hostiles
PASO_MS = 2.0  # Resolución del escaneo (cada cuántos ms se mueve la ventana)
SNR_UMBRAL_SCAN = 100  # Solo forzar escaneo completo si la SNR es <= este valor

if original_snr <= SNR_UMBRAL_SCAN:
    print("\n" + "🔬" * 30)
    print(f"  ESCANEO COMPLETO ACTIVADO (SNR={original_snr} dB <= {SNR_UMBRAL_SCAN} dB)")
    print("🔬" * 30)
    
    paso_idx = int((PASO_MS / 1000.0) * FS)
    L_total = iq_tensor.shape[1]
    
    t_scan_list = []
    prob_list = []
    attn_iq_list = []
    attn_psd_list = []
    
    n_ventanas = (L_total - window_len) // paso_idx + 1
    
    with torch.no_grad():
        for v in range(n_ventanas):
            start_v = v * paso_idx
            end_v = start_v + window_len
            if end_v > L_total:
                break
                
            win_v = iq_tensor[:, start_v:end_v].clone()
            
            # Normalización RMS
            pwr = win_v.pow(2).mean().clamp(min=1e-12).sqrt()
            win_v = win_v / pwr
            
            input_v = win_v.unsqueeze(0).to(device)
            # Usar el z_peak máximo del fichero (si hay detecciones CFAR) como señal global
            z_global = max(z_peaks) if len(z_peaks) > 0 else 0.0
            feat_v = torch.tensor([[global_nf, global_H_mean, z_global]], dtype=torch.float32).to(device)
            
            with torch.amp.autocast('cuda'):
                logit_v, attn_v = model(input_v, feat_v)
                p = torch.sigmoid(logit_v).item() * 100
                aw = attn_v[0].cpu().numpy() * 100
            
            t_centro_ms = ((start_v + window_len / 2) / FS) * 1000
            t_scan_list.append(t_centro_ms)
            prob_list.append(p)
            attn_iq_list.append(aw[0])
            attn_psd_list.append(aw[1])
    
    prob_arr = np.array(prob_list)
    t_arr = np.array(t_scan_list)
    
    # Gráfica de probabilidad temporal
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    
    # Panel 1: Mapa de probabilidad
    ax1.fill_between(t_arr, prob_arr, alpha=0.3, color='red')
    ax1.plot(t_arr, prob_arr, color='red', linewidth=1.5, label='P(Dron)')
    ax1.axhline(y=50, color='gray', linestyle='--', alpha=0.7, label='Umbral 50%')
    ax1.set_ylabel('Probabilidad Dron (%)')
    ax1.set_title(f'Mapa de Probabilidad Temporal — {filename_random} (SNR={original_snr} dB)')
    ax1.set_ylim(0, 100)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Marcar zonas donde P > 50%
    above = prob_arr > 50
    if np.any(above):
        for start_i in np.where(np.diff(above.astype(int)) == 1)[0]:
            end_i_arr = np.where(np.diff(above.astype(int)) == -1)[0]
            end_i = end_i_arr[end_i_arr > start_i]
            if len(end_i) > 0:
                ax1.axvspan(t_arr[start_i], t_arr[end_i[0]], alpha=0.15, color='red')
    
    # Panel 2: Atención IQ vs PSD
    ax2.plot(t_arr, attn_iq_list, color='blue', linewidth=1.2, label='Rama IQ')
    ax2.plot(t_arr, attn_psd_list, color='orange', linewidth=1.2, label='Rama PSD')
    ax2.set_ylabel('Peso Atención (%)')
    ax2.set_xlabel('Tiempo (ms)')
    ax2.set_ylim(0, 100)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    pct_positivo = np.mean(prob_arr > 50) * 100
    p_max = np.max(prob_arr)
    t_max = t_arr[np.argmax(prob_arr)]
    p_mean = np.mean(prob_arr)
    
    print(f"\n📊 ESTADÍSTICAS DEL ESCANEO COMPLETO:")
    print(f"   Ventanas analizadas: {len(prob_arr)}")
    print(f"   Prob. media: {p_mean:.1f}% | Prob. máxima: {p_max:.1f}% (en t={t_max:.1f} ms)")
    print(f"   Ventanas positivas (>50%): {pct_positivo:.1f}%")
    
    if p_max > 70:
        print(f"   🔴 CONFIANZA ALTA: Hay transmisión de dron con pico de {p_max:.0f}%")
    elif p_max > 50:
        print(f"   🟡 CONFIANZA MEDIA: Posible dron (pico {p_max:.0f}%)")
    else:
        print(f"   🟢 CONFIANZA BAJA: Probablemente ruido (pico {p_max:.0f}%)")
else:
    print(f"\n(Escaneo completo no activado: SNR={original_snr} dB > {SNR_UMBRAL_SCAN} dB)")


# Prueba VS modelo alumn V1

In [ ]:
# IMPORTANTE: Cargamos el modelo TEACHER (Oráculo entrenado en SNR >= 0)
ruta_pesos = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_v1\checkpoints\alumn_model_best.pt"
ckpt = torch.load(ruta_pesos, map_location=device, weights_only=False)
model = BurstCVCNN().to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Rescatamos las estadísticas físicas (mean/std) originales con las que se entrenó el Teacher
phys_mean = torch.tensor(ckpt['phys_mean'], dtype=torch.float32).to(device)
phys_std  = torch.tensor(ckpt['phys_std'],  dtype=torch.float32).to(device)
print(f"✅ Nuevo modelo ALUMN cargado con éxito en {device.type.upper()}")
print(f"   Época Óptima del guardado: {ckpt.get('epoch', 'N/A')}")
print(f"   Validation F1: {ckpt.get('val_f1', 0.0):.4f}")

In [ ]:
print("==================================================")
print("     VEREDICTO ALUMNO (Inferencia en vivo)        ")
print("==================================================")
drones_encontrados = 0
ruidos_encontrados = 0

if len(bursts) == 0:
    print("  ❌ No se detectaron ráfagas. La sala se considera VACÍA (RUIDO).")
else:
    # --- AJUSTE DE CLIPPING (Para ser idéntico al entrenamiento) ---
    global_nf      = float(np.clip(np.median(nf_v), 0, 15))
    global_ns_val  = float(np.clip(ns, 0, 5))
    global_H_mean  = float(np.clip(np.mean(H_smooth), 0, 15))
    global_p75_act = float(np.clip(np.percentile(n_active, 75), 0, 2048))
    
    with torch.no_grad():
        for i, b in enumerate(bursts):
            # 1. RECORTAR Y NORMALIZAR ONDA
            idx_inicio = int(b['t0'] * 1e-3 * FS)
            idx_fin = int(b['t1'] * 1e-3 * FS)
            if idx_inicio >= idx_fin: continue
            
            pulso = iq_tensor[:, idx_inicio:idx_fin]
            power = pulso.pow(2).mean().clamp(min=1e-12).sqrt()
            pulso_normalizado = pulso / power
            
            # PADDING ESTÁNDAR (131072 muestras = 9.4 ms)
            TARGET_LEN = 131072
            C, L = pulso_normalizado.shape
            if L < TARGET_LEN:
                pad = torch.zeros(C, TARGET_LEN - L, device=pulso_normalizado.device)
                pulso_padded = torch.cat([pulso_normalizado, pad], dim=1)
            else:
                pulso_padded = pulso_normalizado[:, :TARGET_LEN]
                
            input_ia = pulso_padded.unsqueeze(0).to(device) 
            
            # 2. CONSTRUIR EL PERFIL FÍSICO (8 Dimensiones en orden exacto)
            feat_array = np.array([
                np.clip(b['dur_ms'], 0, 75),
                np.clip(abs(b['z_peak']), 0, 30),
                np.clip(b['drop_b'], 0, 10),
                np.clip(b['n_act'], 0, 2048),
                global_nf, 
                global_ns_val, 
                global_H_mean, 
                global_p75_act
            ], dtype=np.float32)
            
            feat_t = torch.from_numpy(feat_array).to(device)
            # Normalización manual con stats del checkpoint
            feat_norm = torch.clamp((feat_t - phys_mean) / (phys_std + 1e-8), -5.0, 5.0).unsqueeze(0)
            
            # 3. CLASIFICACIÓN
            logit = model(input_ia, feat_norm)
            prob_dron = torch.sigmoid(logit).item() * 100 
            
            t_ms_inicio = b['t0']
            if prob_dron > 50.0:
                drones_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
            else:
                ruidos_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")

print("-" * 50)
print(f"RESUMEN: {drones_encontrados} Drones | {ruidos_encontrados} Ruidos")